In [1]:
import pandas as pd

In [2]:
# Loading dataset
detection_data = pd.read_csv('/home/wolf6273/motion_journal_data_analysis/detection_data.csv', index_col=0)
# detection_data = detection_data[detection_data['split'] != 'valid']
detection_data

,subject,split,latent_var,value,geom_method,geom_dims,temp_method,temp_dims,status
1,1,test,X0,0.242768,GDL,16,TimeVAE,24,iMI
2,1,test,X1,0.145817,GDL,16,TimeVAE,24,iMI
3,1,test,X2,0.357889,GDL,16,TimeVAE,24,iMI
4,1,test,X3,0.481672,GDL,16,TimeVAE,24,iMI
5,1,test,X4,-0.761130,GDL,16,TimeVAE,24,iMI
...,...,...,...,...,...,...,...,...,...
550780,176,test,X7_ES,-571.812262,PCA_COMBINED,12,NONE,0,pMI
550781,176,test,X8_ES,163.007031,PCA_COMBINED,12,NONE,0,pMI
550782,176,test,X9_ES,169.223776,PCA_COMBINED,12,NONE,0,pMI
550783,176,test,X10_ES,-319.460010,PCA_COMBINED,12,NONE,0,pMI


In [3]:
def _get_dataset(status, geom_method, geom_dims, temp_method, temp_dims, df=detection_data):
    """
    Filters the dataset based on the provided parameters.
    
    Args:
        df (pd.DataFrame): The dataset to filter.
        status (str): The status to filter by.
        geom_method (str): The geometric method to filter by.
        geom_dims (int): The number of dimensions for geometry.
        temp_method (str): The temporal method to filter by.
        temp_dims (int): The number of dimensions for time.
    
    Returns:
        pd.DataFrame: Filtered dataset.
    """
    df = df[(df['status'].isin([status, 'healthy'])) &
              (df['geom_method'] == geom_method) &
              (df['geom_dims'] == geom_dims) &
              (df['temp_method'] == temp_method) &
              (df['temp_dims'] == temp_dims)]
    

    df = df.pivot_table(index = ['subject', 'status'], columns='latent_var', values = 'value')

    
    X = df[[col for col in df.columns if col.startswith("X")]].copy()
    y = (df.index.get_level_values('status') != 'healthy').astype(int)
    print(X.shape)

    # Reshape X
    return X, y

In [4]:
import sklearn

from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC

In [5]:
import statsmodels.api as sm
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

In [6]:
class AICForwardSelector(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.selected_features_ = []

    def fit(self, X, y):
        X_df = pd.DataFrame(X, columns=[f"X{i}" for i in range(X.shape[1])])
        X_df['intercept'] = 1
        y_series = pd.Series(y)

        remaining = set(X_df.columns) - {'intercept'}
        selected = []
        current_score, best_new_score = np.inf, np.inf

        while remaining:
            scores_with_candidates = []
            for candidate in remaining:
                formula = 'y ~ ' + ' + '.join(selected + [candidate])
                model = sm.Logit(y_series, X_df[selected + [candidate] + ['intercept']])
                try:
                    result = model.fit(disp=0)
                    scores_with_candidates.append((result.aic, candidate))
                except:
                    continue

            if not scores_with_candidates:
                break

            scores_with_candidates.sort()
            best_new_score, best_candidate = scores_with_candidates[0]

            if best_new_score < current_score:
                remaining.remove(best_candidate)
                selected.append(best_candidate)
                current_score = best_new_score
            else:
                break

        self.selected_features_ = selected
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=[f"X{i}" for i in range(X.shape[1])])
        return X_df[self.selected_features_].values

In [7]:
# Define model pipelines
pipelines = dict()

# No regularisation (requires solver='saga' for penalty='none')
pipelines['none'] = Pipeline([
    ('scaling', StandardScaler()),
    ('classification', LogisticRegression(solver='saga', max_iter=10000))
])

# List of C values to try
c_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
# c_values = [1.0]

for c in c_values:
    name = f"l1_C{c}"
    pipelines[name] = Pipeline([
        ('scaling', StandardScaler()),
        ('classification', LogisticRegression(solver='liblinear', penalty='l1', C=c))
    ])

pipelines['AIC'] = Pipeline([
    ('scale', StandardScaler()),
    ('select', AICForwardSelector()),
    ('logistic', LogisticRegression(solver='liblinear'))
])

# c_values = [0.1]
c_values = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

for c in c_values:
    name = f"l2_C{c}"
    pipelines[name] = Pipeline([
        ('scaling', StandardScaler()),
        ('classification', LogisticRegression(solver='liblinear', penalty='l2', C=c))
    ])

for c in c_values:
    name = f"elastic_C{c}"
    pipelines[name] = Pipeline([
        ('scaling', StandardScaler()),
        ('classification', LogisticRegression(solver='saga', penalty='elasticnet', C=c, l1_ratio=0.2))
    ])

In [8]:
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def print_results(label, results):
    print(f"\n📊 {label} Performance:")
    for metric in scoring:
        scores = results[f'test_{metric}']
        print(f"{metric.upper():<10}: {scores.mean():.3f} ± {scores.std():.3f}")

In [9]:
def fit_models(status, geom_method, geom_dims, temp_method, temp_dims, df=detection_data, pipelines=pipelines):
    """
    Fits models based on the provided parameters and prints their performance.
    
    Args:
        status (str): The status to filter by.
        geom_method (str): The geometric method to filter by.
        geom_dims (int): The number of dimensions for geometry.
        temp_method (str): The temporal method to filter by.
        temp_dims (int): The number of dimensions for time.
        df (pd.DataFrame): The dataset to filter and fit models on.
    """
    X, y = _get_dataset(status, geom_method, geom_dims, temp_method, temp_dims, df)
    
    for label, model in pipelines.items():
        print(f"\n🔍 Fitting model: {label} with {status}, {geom_method}, {geom_dims}, {temp_method}, {temp_dims}")
        results = cross_validate(model, X, y, cv=cv, scoring=scoring)
        print_results(label, results)
    
    return results

In [10]:
results = {}

## BENCHMARK

Benchmark (PCA. TODO: EF)

In [11]:
fit_models('iMI', 'PCA_ED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})

(656, 12)

🔍 Fitting model: none with iMI, PCA_ED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.800 ± 0.011
PRECISION : 0.357 ± 0.373
RECALL    : 0.031 ± 0.029
F1        : 0.054 ± 0.047
ROC_AUC   : 0.638 ± 0.042


/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'fit_time': array([0.00308871, 0.00297213, 0.00251889, 0.00212407, 0.00241494]),
 'score_time': array([0.00412178, 0.00439739, 0.00382829, 0.00397682, 0.00425839]),
 'test_accuracy': array([0.81060606, 0.80916031, 0.80152672, 0.77862595, 0.80152672]),
 'test_precision': array([1.        , 0.5       , 0.        , 0.28571429, 0.        ]),
 'test_recall': array([0.03846154, 0.04      , 0.        , 0.07692308, 0.        ]),
 'test_f1': array([0.07407407, 0.07407407, 0.        , 0.12121212, 0.        ]),
 'test_roc_auc': array([0.67706821, 0.6890566 , 0.59433962, 0.58534799, 0.64652015])}

In [12]:
fit_models('iMI', 'PCA_ES', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})

(656, 12)

🔍 Fitting model: none with iMI, PCA_ES, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.808 ± 0.011
PRECISION : 0.575 ± 0.384
RECALL    : 0.062 ± 0.039
F1        : 0.107 ± 0.063
ROC_AUC   : 0.648 ± 0.027


/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'fit_time': array([0.00427008, 0.00452113, 0.00354743, 0.00303841, 0.00530863]),
 'score_time': array([0.00385094, 0.0038631 , 0.00388551, 0.0037086 , 0.00368953]),
 'test_accuracy': array([0.81818182, 0.80916031, 0.80916031, 0.78625954, 0.81679389]),
 'test_precision': array([1.   , 0.5  , 0.   , 0.375, 1.   ]),
 'test_recall': array([0.07692308, 0.04      , 0.        , 0.11538462, 0.07692308]),
 'test_f1': array([0.14285714, 0.07407407, 0.        , 0.17647059, 0.14285714]),
 'test_roc_auc': array([0.66291727, 0.69358491, 0.62      , 0.62490842, 0.63772894])}

In [13]:
fit_models('pMI', 'PCA_ED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})

(704, 12)

🔍 Fitting model: none with pMI, PCA_ED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.750 ± 0.024
PRECISION : 0.532 ± 0.091
RECALL    : 0.193 ± 0.032
F1        : 0.277 ± 0.029
ROC_AUC   : 0.739 ± 0.054


{'fit_time': array([0.00265241, 0.00236297, 0.00240064, 0.00212383, 0.00242543]),
 'score_time': array([0.00380325, 0.00418806, 0.00374746, 0.00370312, 0.00370312]),
 'test_accuracy': array([0.75886525, 0.75886525, 0.76595745, 0.70212766, 0.76428571]),
 'test_precision': array([0.53333333, 0.53846154, 0.6       , 0.36363636, 0.625     ]),
 'test_recall': array([0.22857143, 0.2       , 0.17142857, 0.22222222, 0.14285714]),
 'test_f1': array([0.32      , 0.29166667, 0.26666667, 0.27586207, 0.23255814]),
 'test_roc_auc': array([0.66037736, 0.76280323, 0.82398922, 0.71084656, 0.7355102 ])}

In [14]:
fit_models('iMI', 'PCA_COMBINED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})

(656, 24)

🔍 Fitting model: none with iMI, PCA_COMBINED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.805 ± 0.009
PRECISION : 0.595 ± 0.211
RECALL    : 0.094 ± 0.046
F1        : 0.152 ± 0.065
ROC_AUC   : 0.663 ± 0.034


{'fit_time': array([0.04729795, 0.04859209, 0.04776478, 0.01892042, 0.0424099 ]),
 'score_time': array([0.00444412, 0.03890538, 0.00374794, 0.00373578, 0.0037055 ]),
 'test_accuracy': array([0.81060606, 0.81679389, 0.80916031, 0.79389313, 0.79389313]),
 'test_precision': array([1.        , 0.6       , 0.5       , 0.44444444, 0.42857143]),
 'test_recall': array([0.03846154, 0.12      , 0.04      , 0.15384615, 0.11538462]),
 'test_f1': array([0.07407407, 0.2       , 0.07407407, 0.22857143, 0.18181818]),
 'test_roc_auc': array([0.71335269, 0.68603774, 0.65169811, 0.64578755, 0.61684982])}

In [15]:
fit_models('pMI', 'PCA_COMBINED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})

(704, 24)

🔍 Fitting model: none with pMI, PCA_COMBINED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.794 ± 0.024
PRECISION : 0.700 ± 0.125
RECALL    : 0.341 ± 0.061
F1        : 0.451 ± 0.059
ROC_AUC   : 0.768 ± 0.044


{'fit_time': array([0.04349518, 0.04170871, 0.04504275, 0.01945567, 0.03959394]),
 'score_time': array([0.00398946, 0.00375533, 0.00374126, 0.00385427, 0.00371265]),
 'test_accuracy': array([0.78014184, 0.80141844, 0.82978723, 0.75886525, 0.8       ]),
 'test_precision': array([0.66666667, 0.70588235, 0.92307692, 0.53846154, 0.66666667]),
 'test_recall': array([0.22857143, 0.34285714, 0.34285714, 0.38888889, 0.4       ]),
 'test_f1': array([0.34042553, 0.46153846, 0.5       , 0.4516129 , 0.5       ]),
 'test_roc_auc': array([0.71132075, 0.8       , 0.83342318, 0.73492063, 0.75972789])}

## PCAxPCA

In [16]:
fit_models('iMI', 'PCA', 12, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})

(656, 50)

🔍 Fitting model: l2_C0.01 with iMI, PCA, 12, PCA, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.805 ± 0.024
PRECISION : 0.519 ± 0.153
RECALL    : 0.172 ± 0.042
F1        : 0.258 ± 0.065
ROC_AUC   : 0.695 ± 0.054


{'fit_time': array([0.00202155, 0.00181723, 0.00189662, 0.00183678, 0.00177693]),
 'score_time': array([0.00418425, 0.00393176, 0.00391293, 0.003865  , 0.00387907]),
 'test_accuracy': array([0.76515152, 0.83969466, 0.80916031, 0.80916031, 0.80152672]),
 'test_precision': array([0.27272727, 0.75      , 0.5       , 0.57142857, 0.5       ]),
 'test_recall': array([0.11538462, 0.24      , 0.16      , 0.15384615, 0.19230769]),
 'test_f1': array([0.16216216, 0.36363636, 0.24242424, 0.24242424, 0.27777778]),
 'test_roc_auc': array([0.61719884, 0.78037736, 0.68641509, 0.66959707, 0.72051282])}

In [17]:
fit_models('iMI', 'PCA', 24, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})

(656, 50)

🔍 Fitting model: l2_C0.001 with iMI, PCA, 24, PCA, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.794 ± 0.025
PRECISION : 0.459 ± 0.152
RECALL    : 0.156 ± 0.035
F1        : 0.230 ± 0.053
ROC_AUC   : 0.679 ± 0.056


{'fit_time': array([0.00180864, 0.00167465, 0.00168538, 0.00166059, 0.00166941]),
 'score_time': array([0.00390005, 0.00386786, 0.00375175, 0.00376177, 0.0037775 ]),
 'test_accuracy': array([0.75757576, 0.80916031, 0.83206107, 0.79389313, 0.77862595]),
 'test_precision': array([0.25      , 0.5       , 0.71428571, 0.44444444, 0.38461538]),
 'test_recall': array([0.11538462, 0.12      , 0.2       , 0.15384615, 0.19230769]),
 'test_f1': array([0.15789474, 0.19354839, 0.3125    , 0.22857143, 0.25641026]),
 'test_roc_auc': array([0.59542816, 0.75849057, 0.72075472, 0.66153846, 0.65824176])}

In [18]:
fit_models('pMI', 'PCA', 12, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})

(704, 50)

🔍 Fitting model: none with pMI, PCA, 12, PCA, 50

📊 none Performance:
ACCURACY  : 0.774 ± 0.053
PRECISION : 0.563 ± 0.148
RECALL    : 0.403 ± 0.122
F1        : 0.469 ± 0.133
ROC_AUC   : 0.792 ± 0.069


{'fit_time': array([0.05191159, 0.07341218, 0.08577394, 0.05797505, 0.09673357]),
 'score_time': array([0.00402164, 0.00393391, 0.00378656, 0.00378299, 0.00383401]),
 'test_accuracy': array([0.73049645, 0.81560284, 0.82978723, 0.80141844, 0.69285714]),
 'test_precision': array([0.42857143, 0.69565217, 0.7037037 , 0.64285714, 0.34615385]),
 'test_recall': array([0.25714286, 0.45714286, 0.54285714, 0.5       , 0.25714286]),
 'test_f1': array([0.32142857, 0.55172414, 0.61290323, 0.5625    , 0.29508197]),
 'test_roc_auc': array([0.76091644, 0.85390836, 0.87897574, 0.78121693, 0.68435374])}

In [19]:
fit_models('pMI', 'PCA', 24, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})

(704, 50)

🔍 Fitting model: l2_C0.001 with pMI, PCA, 24, PCA, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.805 ± 0.046
PRECISION : 0.647 ± 0.116
RECALL    : 0.500 ± 0.079
F1        : 0.564 ± 0.093
ROC_AUC   : 0.802 ± 0.047


{'fit_time': array([0.00183582, 0.00170922, 0.00167823, 0.00169587, 0.00170016]),
 'score_time': array([0.00391269, 0.00388789, 0.00378895, 0.0037508 , 0.00375485]),
 'test_accuracy': array([0.79432624, 0.84397163, 0.86524823, 0.78723404, 0.73571429]),
 'test_precision': array([0.61538462, 0.76      , 0.78571429, 0.60714286, 0.46666667]),
 'test_recall': array([0.45714286, 0.54285714, 0.62857143, 0.47222222, 0.4       ]),
 'test_f1': array([0.52459016, 0.63333333, 0.6984127 , 0.53125   , 0.43076923]),
 'test_roc_auc': array([0.77843666, 0.85040431, 0.86469003, 0.76137566, 0.75319728])}

# PCAxPCAS

In [20]:
fit_models('iMI', 'PCA', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l1_C1.0']})

(656, 71)

🔍 Fitting model: l1_C1.0 with iMI, PCA, 12, PCAS, 50

📊 l1_C1.0 Performance:
ACCURACY  : 0.784 ± 0.041
PRECISION : 0.410 ± 0.184
RECALL    : 0.196 ± 0.082
F1        : 0.263 ± 0.112
ROC_AUC   : 0.704 ± 0.057


{'fit_time': array([0.00429082, 0.00401139, 0.00430775, 0.00528574, 0.00391006]),
 'score_time': array([0.00400496, 0.00389409, 0.00393319, 0.00387764, 0.00387955]),
 'test_accuracy': array([0.75757576, 0.85496183, 0.77862595, 0.73282443, 0.79389313]),
 'test_precision': array([0.28571429, 0.75      , 0.33333333, 0.23529412, 0.44444444]),
 'test_recall': array([0.15384615, 0.36      , 0.16      , 0.15384615, 0.15384615]),
 'test_f1': array([0.2       , 0.48648649, 0.21621622, 0.18604651, 0.22857143]),
 'test_roc_auc': array([0.63352685, 0.80264151, 0.67660377, 0.68205128, 0.72454212])}

In [21]:
fit_models('iMI', 'PCA', 24, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})

(656, 172)

🔍 Fitting model: l2_C0.001 with iMI, PCA, 24, PCAS, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.730 ± 0.065
PRECISION : 0.350 ± 0.118
RECALL    : 0.376 ± 0.084
F1        : 0.359 ± 0.099
ROC_AUC   : 0.706 ± 0.061


{'fit_time': array([0.00349379, 0.00338984, 0.00347972, 0.00315738, 0.00333285]),
 'score_time': array([0.00441813, 0.0044117 , 0.00442719, 0.00445151, 0.00428653]),
 'test_accuracy': array([0.65151515, 0.81679389, 0.73282443, 0.66412214, 0.78625954]),
 'test_precision': array([0.20588235, 0.52      , 0.30769231, 0.26315789, 0.45454545]),
 'test_recall': array([0.26923077, 0.52      , 0.32      , 0.38461538, 0.38461538]),
 'test_f1': array([0.23333333, 0.52      , 0.31372549, 0.3125    , 0.41666667]),
 'test_roc_auc': array([0.6063135 , 0.78981132, 0.7109434 , 0.68571429, 0.73919414])}

In [22]:
fit_models('pMI', 'PCA', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})

(704, 71)

🔍 Fitting model: l2_C0.01 with pMI, PCA, 12, PCAS, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.784 ± 0.036
PRECISION : 0.593 ± 0.094
RECALL    : 0.460 ± 0.057
F1        : 0.517 ± 0.068
ROC_AUC   : 0.798 ± 0.055


{'fit_time': array([0.00239825, 0.00226355, 0.00225115, 0.00224543, 0.00242829]),
 'score_time': array([0.00421166, 0.00383186, 0.00383973, 0.00384665, 0.0038619 ]),
 'test_accuracy': array([0.76595745, 0.83687943, 0.81560284, 0.74468085, 0.75714286]),
 'test_precision': array([0.53571429, 0.73076923, 0.68      , 0.5       , 0.52      ]),
 'test_recall': array([0.42857143, 0.54285714, 0.48571429, 0.47222222, 0.37142857]),
 'test_f1': array([0.47619048, 0.62295082, 0.56666667, 0.48571429, 0.43333333]),
 'test_roc_auc': array([0.79380054, 0.8425876 , 0.87466307, 0.7468254 , 0.73034014])}

In [23]:
fit_models('pMI', 'PCA', 24, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})

(704, 172)

🔍 Fitting model: l2_C0.001 with pMI, PCA, 24, PCAS, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.764 ± 0.035
PRECISION : 0.526 ± 0.054
RECALL    : 0.665 ± 0.031
F1        : 0.587 ± 0.044
ROC_AUC   : 0.795 ± 0.043


{'fit_time': array([0.00371552, 0.00346398, 0.0034399 , 0.00317168, 0.00345254]),
 'score_time': array([0.00455427, 0.00455117, 0.00423431, 0.00424433, 0.0042665 ]),
 'test_accuracy': array([0.73049645, 0.81560284, 0.78723404, 0.72340426, 0.76428571]),
 'test_precision': array([0.46938776, 0.6097561 , 0.55813953, 0.46938776, 0.52380952]),
 'test_recall': array([0.65714286, 0.71428571, 0.68571429, 0.63888889, 0.62857143]),
 'test_f1': array([0.54761905, 0.65789474, 0.61538462, 0.54117647, 0.57142857]),
 'test_roc_auc': array([0.81590296, 0.85229111, 0.81536388, 0.75714286, 0.73442177])}

## GDLxPCAS

In [24]:
fit_models('iMI', 'GDL', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})

(656, 63)

🔍 Fitting model: l2_C0.01 with iMI, GDL, 12, PCAS, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.785 ± 0.027
PRECISION : 0.403 ± 0.148
RECALL    : 0.133 ± 0.039
F1        : 0.194 ± 0.051
ROC_AUC   : 0.680 ± 0.075


{'fit_time': array([0.00236869, 0.00219798, 0.00242972, 0.00220156, 0.00221252]),
 'score_time': array([0.0039711 , 0.00379395, 0.0038507 , 0.00386906, 0.0038352 ]),
 'test_accuracy': array([0.74242424, 0.80916031, 0.81679389, 0.77099237, 0.78625954]),
 'test_precision': array([0.16666667, 0.5       , 0.6       , 0.33333333, 0.41666667]),
 'test_recall': array([0.07692308, 0.12      , 0.12      , 0.15384615, 0.19230769]),
 'test_f1': array([0.10526316, 0.19354839, 0.2       , 0.21052632, 0.26315789]),
 'test_roc_auc': array([0.59760522, 0.81886792, 0.68603774, 0.63479853, 0.66300366])}

In [25]:
fit_models('pMI', 'GDL', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})

(704, 63)

🔍 Fitting model: l2_C0.01 with pMI, GDL, 12, PCAS, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.793 ± 0.044
PRECISION : 0.601 ± 0.101
RECALL    : 0.512 ± 0.086
F1        : 0.553 ± 0.092
ROC_AUC   : 0.802 ± 0.061


{'fit_time': array([0.00244141, 0.00217915, 0.00229597, 0.00225806, 0.00226784]),
 'score_time': array([0.00410986, 0.00381136, 0.00381708, 0.00380993, 0.00384092]),
 'test_accuracy': array([0.78014184, 0.82269504, 0.85815603, 0.77304965, 0.72857143]),
 'test_precision': array([0.56666667, 0.67857143, 0.74193548, 0.56666667, 0.4516129 ]),
 'test_recall': array([0.48571429, 0.54285714, 0.65714286, 0.47222222, 0.4       ]),
 'test_f1': array([0.52307692, 0.6031746 , 0.6969697 , 0.51515152, 0.42424242]),
 'test_roc_auc': array([0.77816712, 0.84878706, 0.89541779, 0.75529101, 0.73142857])}

## PCAxTimeVAE

In [26]:
fit_models('iMI', 'PCA', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.1']})

(656, 24)

🔍 Fitting model: l2_C0.1 with iMI, PCA, 16, TimeVAE, 24

📊 l2_C0.1 Performance:
ACCURACY  : 0.805 ± 0.010
PRECISION : 0.580 ± 0.238
RECALL    : 0.070 ± 0.015
F1        : 0.123 ± 0.025
ROC_AUC   : 0.676 ± 0.043


{'fit_time': array([0.00191808, 0.00171399, 0.00175309, 0.00171828, 0.0017159 ]),
 'score_time': array([0.00381017, 0.00368738, 0.00364876, 0.00397444, 0.00366855]),
 'test_accuracy': array([0.81060606, 0.80152672, 0.80916031, 0.81679389, 0.78625954]),
 'test_precision': array([0.66666667, 0.4       , 0.5       , 1.        , 0.33333333]),
 'test_recall': array([0.07692308, 0.08      , 0.04      , 0.07692308, 0.07692308]),
 'test_f1': array([0.13793103, 0.13333333, 0.07407407, 0.14285714, 0.125     ]),
 'test_roc_auc': array([0.65493469, 0.73886792, 0.6354717 , 0.71684982, 0.63589744])}

In [27]:
fit_models('pMI', 'PCA', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.1']})

(704, 24)

🔍 Fitting model: l2_C0.1 with pMI, PCA, 16, TimeVAE, 24

📊 l2_C0.1 Performance:
ACCURACY  : 0.757 ± 0.011
PRECISION : 0.533 ± 0.048
RECALL    : 0.301 ± 0.041
F1        : 0.381 ± 0.030
ROC_AUC   : 0.766 ± 0.023


{'fit_time': array([0.0019865 , 0.00178766, 0.0017705 , 0.00176811, 0.00178719]),
 'score_time': array([0.00396347, 0.00368905, 0.00367379, 0.0036571 , 0.00368428]),
 'test_accuracy': array([0.77304965, 0.74468085, 0.75886525, 0.74468085, 0.76428571]),
 'test_precision': array([0.61538462, 0.47619048, 0.52173913, 0.5       , 0.55      ]),
 'test_recall': array([0.22857143, 0.28571429, 0.34285714, 0.33333333, 0.31428571]),
 'test_f1': array([0.33333333, 0.35714286, 0.4137931 , 0.4       , 0.4       ]),
 'test_roc_auc': array([0.74663073, 0.80026954, 0.78328841, 0.73756614, 0.76136054])}

## GDLxTimeVAE

In [28]:
fit_models('iMI', 'GDL', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C1.0']})

(656, 24)

🔍 Fitting model: l2_C1.0 with iMI, GDL, 16, TimeVAE, 24

📊 l2_C1.0 Performance:
ACCURACY  : 0.794 ± 0.017
PRECISION : 0.426 ± 0.157
RECALL    : 0.086 ± 0.028
F1        : 0.139 ± 0.044
ROC_AUC   : 0.665 ± 0.044


{'fit_time': array([0.00220227, 0.00192928, 0.00211644, 0.00212979, 0.00209188]),
 'score_time': array([0.00398779, 0.00364947, 0.00363374, 0.00367856, 0.003685  ]),
 'test_accuracy': array([0.79545455, 0.81679389, 0.76335878, 0.79389313, 0.80152672]),
 'test_precision': array([0.33333333, 0.66666667, 0.2       , 0.42857143, 0.5       ]),
 'test_recall': array([0.03846154, 0.08      , 0.08      , 0.11538462, 0.11538462]),
 'test_f1': array([0.06896552, 0.14285714, 0.11428571, 0.18181818, 0.1875    ]),
 'test_roc_auc': array([0.6988389 , 0.73433962, 0.63132075, 0.64358974, 0.61684982])}

In [29]:
fit_models('pMI', 'GDL', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l1_C1.0']})

(704, 24)

🔍 Fitting model: l1_C1.0 with pMI, GDL, 16, TimeVAE, 24

📊 l1_C1.0 Performance:
ACCURACY  : 0.788 ± 0.023
PRECISION : 0.655 ± 0.119
RECALL    : 0.375 ± 0.101
F1        : 0.464 ± 0.067
ROC_AUC   : 0.790 ± 0.037


{'fit_time': array([0.00504732, 0.00594425, 0.00394869, 0.00479388, 0.00530958]),
 'score_time': array([0.00397158, 0.00368619, 0.00374413, 0.00370526, 0.00365186]),
 'test_accuracy': array([0.80851064, 0.81560284, 0.78723404, 0.78014184, 0.75      ]),
 'test_precision': array([0.78571429, 0.8       , 0.57142857, 0.61904762, 0.5       ]),
 'test_recall': array([0.31428571, 0.34285714, 0.57142857, 0.36111111, 0.28571429]),
 'test_f1': array([0.44897959, 0.48      , 0.57142857, 0.45614035, 0.36363636]),
 'test_roc_auc': array([0.81132075, 0.84663073, 0.7916442 , 0.7531746 , 0.74585034])}

## PCAxTimeVAES

In [30]:
fit_models('iMI', 'PCA', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['l1_C10.0']})

(656, 80)

🔍 Fitting model: l1_C10.0 with iMI, PCA, 16, TimeVAES, 5

📊 l1_C10.0 Performance:
ACCURACY  : 0.782 ± 0.030
PRECISION : 0.414 ± 0.114
RECALL    : 0.235 ± 0.053
F1        : 0.297 ± 0.069
ROC_AUC   : 0.706 ± 0.034


{'fit_time': array([0.0382309 , 0.03488922, 0.03187156, 0.03220367, 0.02917218]),
 'score_time': array([0.00408244, 0.00388861, 0.00441623, 0.00394511, 0.00389743]),
 'test_accuracy': array([0.75      , 0.75572519, 0.83206107, 0.77862595, 0.79389313]),
 'test_precision': array([0.31578947, 0.31578947, 0.61538462, 0.36363636, 0.46153846]),
 'test_recall': array([0.23076923, 0.24      , 0.32      , 0.15384615, 0.23076923]),
 'test_f1': array([0.26666667, 0.27272727, 0.42105263, 0.21621622, 0.30769231]),
 'test_roc_auc': array([0.64259797, 0.72528302, 0.74188679, 0.71062271, 0.70989011])}

In [31]:
fit_models('pMI', 'PCA', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})

(704, 80)

🔍 Fitting model: l2_C0.01 with pMI, PCA, 16, TimeVAES, 5

📊 l2_C0.01 Performance:
ACCURACY  : 0.793 ± 0.013
PRECISION : 0.602 ± 0.035
RECALL    : 0.511 ± 0.048
F1        : 0.551 ± 0.032
ROC_AUC   : 0.792 ± 0.021


{'fit_time': array([0.00303125, 0.00270629, 0.00268865, 0.00280714, 0.00267553]),
 'score_time': array([0.00410151, 0.00391221, 0.00388741, 0.00387335, 0.00397873]),
 'test_accuracy': array([0.79432624, 0.78723404, 0.80851064, 0.80141844, 0.77142857]),
 'test_precision': array([0.58823529, 0.6       , 0.63333333, 0.64285714, 0.54545455]),
 'test_recall': array([0.57142857, 0.42857143, 0.54285714, 0.5       , 0.51428571]),
 'test_f1': array([0.57971014, 0.5       , 0.58461538, 0.5625    , 0.52941176]),
 'test_roc_auc': array([0.78544474, 0.8296496 , 0.79757412, 0.76719577, 0.78013605])}

## GDLxTimeVAES

In [32]:
fit_models('iMI', 'GDL', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.1']})

(656, 80)

🔍 Fitting model: l2_C0.1 with iMI, GDL, 16, TimeVAES, 5

📊 l2_C0.1 Performance:
ACCURACY  : 0.784 ± 0.021
PRECISION : 0.374 ± 0.095
RECALL    : 0.157 ± 0.045
F1        : 0.219 ± 0.060
ROC_AUC   : 0.685 ± 0.042


{'fit_time': array([0.00381374, 0.00366259, 0.00406623, 0.00406146, 0.00411224]),
 'score_time': array([0.00422478, 0.00392675, 0.00393867, 0.00393391, 0.00395775]),
 'test_accuracy': array([0.77272727, 0.80152672, 0.80152672, 0.7480916 , 0.79389313]),
 'test_precision': array([0.25      , 0.45454545, 0.45454545, 0.26666667, 0.44444444]),
 'test_recall': array([0.07692308, 0.2       , 0.2       , 0.15384615, 0.15384615]),
 'test_f1': array([0.11764706, 0.27777778, 0.27777778, 0.19512195, 0.22857143]),
 'test_roc_auc': array([0.61211901, 0.73698113, 0.67245283, 0.6978022 , 0.70549451])}

In [33]:
fit_models('pMI', 'GDL', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['elastic_C0.1']})

(704, 80)

🔍 Fitting model: elastic_C0.1 with pMI, GDL, 16, TimeVAES, 5

📊 elastic_C0.1 Performance:
ACCURACY  : 0.804 ± 0.024
PRECISION : 0.695 ± 0.083
RECALL    : 0.398 ± 0.058
F1        : 0.503 ± 0.059
ROC_AUC   : 0.808 ± 0.020


/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.w

{'fit_time': array([0.02886271, 0.02848816, 0.0284512 , 0.02845073, 0.02873492]),
 'score_time': array([0.00419712, 0.00395536, 0.0039413 , 0.00401115, 0.00407505]),
 'test_accuracy': array([0.81560284, 0.77304965, 0.84397163, 0.79432624, 0.79285714]),
 'test_precision': array([0.73684211, 0.55555556, 0.80952381, 0.68421053, 0.6875    ]),
 'test_recall': array([0.4       , 0.42857143, 0.48571429, 0.36111111, 0.31428571]),
 'test_f1': array([0.51851852, 0.48387097, 0.60714286, 0.47272727, 0.43137255]),
 'test_roc_auc': array([0.78247978, 0.83180593, 0.82318059, 0.81825397, 0.78612245])}

# Combined results

In [34]:
results_iMI = {}
results_pMI = {}

In [35]:
results_iMI['PCA ED'] = fit_models('iMI', 'PCA_ED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})
results_iMI['PCA ES'] = fit_models('iMI', 'PCA_ES', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})
results_pMI['PCA ED'] = fit_models('pMI', 'PCA_ED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})
results_pMI['PCA ES'] = fit_models('pMI', 'PCA_ES', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})

results_iMI['PCA COMBINED'] = fit_models('iMI', 'PCA_COMBINED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})
results_pMI['PCA COMBINED'] = fit_models('pMI', 'PCA_COMBINED', 12, 'NONE', 0, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})


(656, 12)

🔍 Fitting model: none with iMI, PCA_ED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.800 ± 0.011
PRECISION : 0.357 ± 0.373
RECALL    : 0.031 ± 0.029
F1        : 0.054 ± 0.047
ROC_AUC   : 0.639 ± 0.042
(656, 12)

🔍 Fitting model: none with iMI, PCA_ES, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.808 ± 0.011
PRECISION : 0.575 ± 0.384
RECALL    : 0.062 ± 0.039
F1        : 0.107 ± 0.063
ROC_AUC   : 0.648 ± 0.027
(704, 12)

🔍 Fitting model: none with pMI, PCA_ED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.750 ± 0.024
PRECISION : 0.532 ± 0.091
RECALL    : 0.193 ± 0.032
F1        : 0.277 ± 0.029
ROC_AUC   : 0.739 ± 0.054


/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(704, 12)

🔍 Fitting model: none with pMI, PCA_ES, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.770 ± 0.027
PRECISION : 0.599 ± 0.093
RECALL    : 0.278 ± 0.060
F1        : 0.375 ± 0.065
ROC_AUC   : 0.755 ± 0.053
(656, 24)

🔍 Fitting model: none with iMI, PCA_COMBINED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.805 ± 0.009
PRECISION : 0.595 ± 0.211
RECALL    : 0.094 ± 0.046
F1        : 0.152 ± 0.065
ROC_AUC   : 0.663 ± 0.034
(704, 24)

🔍 Fitting model: none with pMI, PCA_COMBINED, 12, NONE, 0

📊 none Performance:
ACCURACY  : 0.794 ± 0.024
PRECISION : 0.700 ± 0.125
RECALL    : 0.341 ± 0.061
F1        : 0.451 ± 0.059
ROC_AUC   : 0.768 ± 0.044


In [36]:
results_iMI['PCAxPCA (12, 50)'] = fit_models('iMI', 'PCA', 12, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})
results_iMI['PCAxPCA (24, 50)'] = fit_models('iMI', 'PCA', 24, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})
results_pMI['PCAxPCA (12, 50)'] = fit_models('pMI', 'PCA', 12, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['none']})
results_pMI['PCAxPCA (24, 50)'] = fit_models('pMI', 'PCA', 24, 'PCA', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})

(656, 50)

🔍 Fitting model: l2_C0.01 with iMI, PCA, 12, PCA, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.805 ± 0.024
PRECISION : 0.519 ± 0.153
RECALL    : 0.172 ± 0.042
F1        : 0.258 ± 0.065
ROC_AUC   : 0.695 ± 0.054
(656, 50)

🔍 Fitting model: l2_C0.001 with iMI, PCA, 24, PCA, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.794 ± 0.025
PRECISION : 0.459 ± 0.152
RECALL    : 0.156 ± 0.035
F1        : 0.230 ± 0.053
ROC_AUC   : 0.679 ± 0.056
(704, 50)

🔍 Fitting model: none with pMI, PCA, 12, PCA, 50

📊 none Performance:
ACCURACY  : 0.774 ± 0.053
PRECISION : 0.563 ± 0.148
RECALL    : 0.403 ± 0.122
F1        : 0.469 ± 0.133
ROC_AUC   : 0.792 ± 0.069
(704, 50)

🔍 Fitting model: l2_C0.001 with pMI, PCA, 24, PCA, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.805 ± 0.046
PRECISION : 0.647 ± 0.116
RECALL    : 0.500 ± 0.079
F1        : 0.564 ± 0.093
ROC_AUC   : 0.802 ± 0.047


In [37]:
results_iMI['PCAxPCAS (12,50)'] = fit_models('iMI', 'PCA', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l1_C1.0']})
results_iMI['PCAxPCAS (24,50)'] = fit_models('iMI', 'PCA', 24, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})
results_pMI['PCAxPCAS (12,50)'] = fit_models('pMI', 'PCA', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})
results_pMI['PCAxPCAS (24,50)'] = fit_models('pMI', 'PCA', 24, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.001']})

(656, 71)

🔍 Fitting model: l1_C1.0 with iMI, PCA, 12, PCAS, 50

📊 l1_C1.0 Performance:
ACCURACY  : 0.784 ± 0.041
PRECISION : 0.410 ± 0.184
RECALL    : 0.196 ± 0.082
F1        : 0.263 ± 0.112
ROC_AUC   : 0.704 ± 0.057
(656, 172)

🔍 Fitting model: l2_C0.001 with iMI, PCA, 24, PCAS, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.730 ± 0.065
PRECISION : 0.350 ± 0.118
RECALL    : 0.376 ± 0.084
F1        : 0.359 ± 0.099
ROC_AUC   : 0.706 ± 0.061
(704, 71)

🔍 Fitting model: l2_C0.01 with pMI, PCA, 12, PCAS, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.784 ± 0.036
PRECISION : 0.593 ± 0.094
RECALL    : 0.460 ± 0.057
F1        : 0.517 ± 0.068
ROC_AUC   : 0.798 ± 0.055
(704, 172)

🔍 Fitting model: l2_C0.001 with pMI, PCA, 24, PCAS, 50

📊 l2_C0.001 Performance:
ACCURACY  : 0.764 ± 0.035
PRECISION : 0.526 ± 0.054
RECALL    : 0.665 ± 0.031
F1        : 0.587 ± 0.044
ROC_AUC   : 0.795 ± 0.043


In [38]:
results_iMI['GDLxPCAS'] = fit_models('iMI', 'GDL', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})
results_pMI['GDLxPCAS'] = fit_models('pMI', 'GDL', 12, 'PCAS', 50, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})


(656, 63)

🔍 Fitting model: l2_C0.01 with iMI, GDL, 12, PCAS, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.785 ± 0.027
PRECISION : 0.403 ± 0.148
RECALL    : 0.133 ± 0.039
F1        : 0.194 ± 0.051
ROC_AUC   : 0.680 ± 0.075
(704, 63)

🔍 Fitting model: l2_C0.01 with pMI, GDL, 12, PCAS, 50

📊 l2_C0.01 Performance:
ACCURACY  : 0.793 ± 0.044
PRECISION : 0.601 ± 0.101
RECALL    : 0.512 ± 0.086
F1        : 0.553 ± 0.092
ROC_AUC   : 0.802 ± 0.061


In [39]:
results_iMI['PCAxTimeVAE'] = fit_models('iMI', 'PCA', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.1']})
results_pMI['PCAxTimeVAE'] = fit_models('pMI', 'PCA', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.1']})
results_iMI['GDLxTimeVAE'] = fit_models('iMI', 'GDL', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C1.0']})
results_pMI['GDLxTimeVAE'] = fit_models('pMI', 'GDL', 16, 'TimeVAE', 24, pipelines={key: pipelines[key] for key in pipelines if key in ['l1_C1.0']})

(656, 24)

🔍 Fitting model: l2_C0.1 with iMI, PCA, 16, TimeVAE, 24

📊 l2_C0.1 Performance:
ACCURACY  : 0.805 ± 0.010
PRECISION : 0.580 ± 0.238
RECALL    : 0.070 ± 0.015
F1        : 0.123 ± 0.025
ROC_AUC   : 0.676 ± 0.043
(704, 24)

🔍 Fitting model: l2_C0.1 with pMI, PCA, 16, TimeVAE, 24

📊 l2_C0.1 Performance:
ACCURACY  : 0.757 ± 0.011
PRECISION : 0.533 ± 0.048
RECALL    : 0.301 ± 0.041
F1        : 0.381 ± 0.030
ROC_AUC   : 0.766 ± 0.023
(656, 24)

🔍 Fitting model: l2_C1.0 with iMI, GDL, 16, TimeVAE, 24

📊 l2_C1.0 Performance:
ACCURACY  : 0.794 ± 0.017
PRECISION : 0.426 ± 0.157
RECALL    : 0.086 ± 0.028
F1        : 0.139 ± 0.044
ROC_AUC   : 0.665 ± 0.044
(704, 24)

🔍 Fitting model: l1_C1.0 with pMI, GDL, 16, TimeVAE, 24

📊 l1_C1.0 Performance:
ACCURACY  : 0.788 ± 0.023
PRECISION : 0.655 ± 0.119
RECALL    : 0.375 ± 0.101
F1        : 0.464 ± 0.067
ROC_AUC   : 0.790 ± 0.037


In [40]:
results_iMI['PCAxTimeVAES'] = fit_models('iMI', 'PCA', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['l1_C10.0']})
results_pMI['PCAxTimeVAES'] = fit_models('pMI', 'PCA', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.01']})
results_iMI['GDLxTimeVAES'] = fit_models('iMI', 'GDL', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['l2_C0.1']})
results_pMI['GDLxTimeVAES'] = fit_models('pMI', 'GDL', 16, 'TimeVAES', 5, pipelines={key: pipelines[key] for key in pipelines if key in ['elastic_C0.1']})

(656, 80)

🔍 Fitting model: l1_C10.0 with iMI, PCA, 16, TimeVAES, 5

📊 l1_C10.0 Performance:
ACCURACY  : 0.782 ± 0.030
PRECISION : 0.414 ± 0.114
RECALL    : 0.235 ± 0.053
F1        : 0.297 ± 0.069
ROC_AUC   : 0.706 ± 0.034
(704, 80)

🔍 Fitting model: l2_C0.01 with pMI, PCA, 16, TimeVAES, 5

📊 l2_C0.01 Performance:
ACCURACY  : 0.793 ± 0.013
PRECISION : 0.602 ± 0.035
RECALL    : 0.511 ± 0.048
F1        : 0.551 ± 0.032
ROC_AUC   : 0.792 ± 0.021
(656, 80)

🔍 Fitting model: l2_C0.1 with iMI, GDL, 16, TimeVAES, 5

📊 l2_C0.1 Performance:
ACCURACY  : 0.784 ± 0.021
PRECISION : 0.374 ± 0.095
RECALL    : 0.157 ± 0.045
F1        : 0.219 ± 0.060
ROC_AUC   : 0.685 ± 0.042
(704, 80)

🔍 Fitting model: elastic_C0.1 with pMI, GDL, 16, TimeVAES, 5

📊 elastic_C0.1 Performance:
ACCURACY  : 0.804 ± 0.024
PRECISION : 0.695 ± 0.083
RECALL    : 0.398 ± 0.058
F1        : 0.503 ± 0.059
ROC_AUC   : 0.808 ± 0.020


/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/wolf6273/miniconda3/envs/myenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.w

In [41]:
for key1 in results_iMI.keys():
    for key2 in results_iMI[key1].keys():
        results_iMI[key1][key2] = np.mean(results_iMI[key1][key2])
results_iMI = pd.DataFrame(results_iMI).transpose()

In [42]:
results_iMI_table = results_iMI[[column for column in results_iMI.columns if column.startswith('test_')]]

In [43]:
results_iMI_table.to_csv('/home/wolf6273/motion_journal_data_analysis/results_iMI.csv')
results_iMI

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
PCA ED,0.002380,0.003730,0.800289,0.357143,0.031077,0.053872,0.638540
PCA ES,0.004227,0.003667,0.807911,0.575000,0.061846,0.107252,0.647828
PCA COMBINED,0.040472,0.003739,0.804869,0.594603,0.093538,0.151708,0.662745
"PCAxPCA (12, 50)",0.001784,0.003809,0.804939,0.518831,0.172308,0.257685,0.694820
"PCAxPCA (24, 50)",0.001692,0.003770,0.794263,0.458669,0.156308,0.229785,0.678891
"PCAxPCAS (12,50)",0.004275,0.003881,0.783576,0.409757,0.196308,0.263464,0.703873
"PCAxPCAS (24,50)",0.003284,0.004225,0.730303,0.350256,0.375692,0.359245,0.706395
GDLxPCAS,0.002235,0.003838,0.785126,0.403333,0.132615,0.194499,0.680063
PCAxTimeVAE,0.001784,0.003743,0.804869,0.580000,0.070154,0.122639,0.676404
GDLxTimeVAE,0.002092,0.003665,0.794205,0.425714,0.085846,0.139085,0.664988


In [44]:
for key1 in results_pMI.keys():
    for key2 in results_pMI[key1].keys():
        results_pMI[key1][key2] = np.mean(results_pMI[key1][key2])
results_pMI = pd.DataFrame(results_pMI).transpose()

In [45]:
results_pMI_table = results_pMI[[column for column in results_pMI.columns if column.startswith('test_')]]

In [46]:
results_pMI_table.to_csv('/home/wolf6273/motion_journal_data_analysis/results_pMI.csv')
results_pMI

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
PCA ED,0.002414,0.003659,0.750020,0.532086,0.193016,0.277351,0.738705
PCA ES,0.004831,0.003616,0.769899,0.599003,0.278254,0.374851,0.754849
PCA COMBINED,0.038396,0.003754,0.794043,0.700151,0.340635,0.450715,0.767825
"PCAxPCA (12, 50)",0.073335,0.003936,0.774032,0.563388,0.402857,0.468728,0.791874
"PCAxPCA (24, 50)",0.001730,0.003799,0.805299,0.646982,0.500159,0.563671,0.801621
"PCAxPCAS (12,50)",0.002323,0.003886,0.784053,0.593297,0.460159,0.516971,0.797643
"PCAxPCAS (24,50)",0.003421,0.004270,0.764205,0.526096,0.664921,0.586701,0.795025
GDLxPCAS,0.002246,0.003860,0.792523,0.601091,0.511587,0.552523,0.801818
PCAxTimeVAE,0.001783,0.003676,0.757112,0.532663,0.300952,0.380854,0.765823
GDLxTimeVAE,0.004814,0.003633,0.788298,0.655238,0.375079,0.464037,0.789777
